# Adım 1 — Veri İndirme ve Tarihlendirme

**Proje:** Erken Cumhuriyet Dönemi TBMM Tutanakları İmla Normalizasyonu  
**Veri:** `boun-tabilab/turkish_parliamentary_data` · `tbmm_only` · term 3–6 · modern_turkish  
**Çıktı:** `data/raw/oturumlar.jsonl` (1316 oturum, %100 kapsama)

## 0. Ortam Kurulumu

**İlk kez çalıştırıyorsan** sırayla çalıştır: bu hücre → kurulum hücresi → devam.

In [ ]:
import sys
sys.path.insert(0, "../src")
sys.path.insert(0, "src")
from ortam import ortam_sec

BASE = ortam_sec()   # Local mi, Drive mı? — sorar ve kaydeder

## 1. Paket Kurulumu (sadece ilk kez)

```bash
# !pip install -r requirements.txt
# !python -c "import nltk; nltk.download('punkt_tab')"
```

In [ ]:
import json
import pandas as pd
from adim1_veri_indir import metinden_tarih_cek, _ocr_normalize, veri_indir_ve_isle

## 2. Tarih Çıkarıcıyı Test Et

Veriyi indirmeden önce 3 formatın çalıştığını doğrulayalım.

In [ ]:
ornekler = [
    ("17 Şubat 1930 Pazartesi",    "Yazılı ay"),
    ("3 Teşrinievvel 1931",         "Osmanlı ay adı"),
    ("15 - XI - 1937 Pazartesi",   "Romen rakamlı"),
    ("20 - 1 - 1937 Çarşamba",     "Romen (OCR: I→1)"),
    ("5.1.1929 Cumartesi",          "Sayısal"),
    ("31 . 12 . 1928",              "Sayısal boşluklu"),
    ("3 Teģrinisāni 1927",          "OCR bozuk karakter"),
]

for metin, aciklama in ornekler:
    tarih, kaynak = metinden_tarih_cek(metin)
    print(f"{aciklama:<22} → {tarih}  [{kaynak}]")

## 3. Veriyi İndir ve İşle

> İlk indirme ~5–10 dakika sürebilir (Hugging Face'ten ~1316 oturum).

In [ ]:
kayitlar = veri_indir_ve_isle()

## 4. Kapsama Raporu

In [ ]:
df = pd.DataFrame(kayitlar)

print(f"Toplam oturum : {len(df)}")
print(f"Tarihi olan   : {df['takvim_yil'].notna().sum()}")
print()

kaynak_ozet = df['date_source'].value_counts().reset_index()
kaynak_ozet.columns = ['date_source', 'sayi']
kaynak_ozet['oran'] = (kaynak_ozet['sayi'] / len(df) * 100).round(1).astype(str) + '%'
print(kaynak_ozet.to_string(index=False))

In [ ]:
# Alt dönem ve term dağılımı
df.groupby(['alt_donem', 'term']).size().reset_index(name='oturum_sayisi')

In [ ]:
# Yıl bazında oturum sayısı
df.groupby('takvim_yil').size().reset_index(name='oturum').sort_values('takvim_yil')

## 5. Örnek Kayıtlar

In [ ]:
# Her date_source'tan birer örnek
for kaynak in ['regex_yazi', 'regex_romen', 'regex_sayisal', 'komşu']:
    ornekler = df[df['date_source'] == kaynak].head(1)
    orneklers = ornekler.to_dict('records')
    if orneklers:
        k = orneklers[0]
        tarih = f"{k['takvim_gun']:02d}.{k['takvim_ay']:02d}.{k['takvim_yil']}" if k['takvim_gun'] else f"~{k['takvim_yil']}"
        print(f"[{kaynak}] {k['document_id']} → {tarih}")
        print(f"  {(k['text'] or '')[:150].replace(chr(10), ' ')}")
        print()

## 6. Manuel Doğrulama Listesi

`guide/dogrulama_adim1.csv` — 30 oturum, PDF ile karşılaştırılacak (YY sorumluluğu).

In [ ]:
dogrulama = pd.read_csv("../guide/dogrulama_adim1.csv")
dogrulama[['no', 'document_id', 'term', 'tarih', 'date_source']].head(10)

---
**Sonuç:** 1316/1316 oturum tarihlendi (%100 kapsama).  
Bir sonraki adım → `adim2_bolme.ipynb` (test/dev/havuz bölmesi, seed=42)